In [4]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [5]:
date = read_table("select * from sc_gold.dim_date")
sector= read_table("select * from sc_gold.dim_sector")
skill_level = read_table("select * from sc_gold.dim_skilllevel")

In [6]:
df = read_table("select * from sc_bronze.dosm_jobdemand")
df 

,date,skill_level,sector,job_available,job_filled,job_vacancy,job_created
0,2018-01-01,Low-skilled,Agriculture,35200.0,26700.0,8500.0,100.0
1,2018-01-01,Low-skilled,Construction,50200.0,45200.0,5000.0,300.0
2,2018-01-01,Low-skilled,Manufacturing,158300.0,140100.0,18200.0,0.0
3,2018-01-01,Low-skilled,Mining and Quarrying,10000.0,9900.0,100.0,0.0
4,2018-01-01,Low-skilled,Services,889100.0,878800.0,10300.0,1700.0
...,...,...,...,...,...,...,...
430,2025-01-01,Skilled,Agriculture,28800.0,24400.0,4400.0,100.0
431,2025-01-01,Skilled,Construction,142400.0,137900.0,4400.0,500.0
432,2025-01-01,Skilled,Manufacturing,444500.0,419100.0,25300.0,2600.0
433,2025-01-01,Skilled,Mining and Quarrying,23500.0,23400.0,100.0,100.0


In [7]:
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    sector[["sector", "sector_id"]],
    on="sector",
    how="left"
)

df = df.merge(
    skill_level[["skill_level", "skill_level_id"]],
    on="skill_level",
    how="left"
)

df_final = df.drop(columns=["date", "sector", "skill_level"])
id_cols = ["date_id", "sector_id", "skill_level_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [8]:
df_final["job_id"] = ["JOB" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["job_id"] + [c for c in df_final.columns if c != "job_id"]]
df_final

,job_id,date_id,sector_id,skill_level_id,job_available,job_filled,job_vacancy,job_created
0,JOB0001,DT009,SEC001,SL001,35200.0,26700.0,8500.0,100.0
1,JOB0002,DT009,SEC002,SL001,50200.0,45200.0,5000.0,300.0
2,JOB0003,DT009,SEC003,SL001,158300.0,140100.0,18200.0,0.0
3,JOB0004,DT009,SEC004,SL001,10000.0,9900.0,100.0,0.0
4,JOB0005,DT009,SEC005,SL001,889100.0,878800.0,10300.0,1700.0
...,...,...,...,...,...,...,...,...
430,JOB0431,DT037,SEC001,SL003,28800.0,24400.0,4400.0,100.0
431,JOB0432,DT037,SEC002,SL003,142400.0,137900.0,4400.0,500.0
432,JOB0433,DT037,SEC003,SL003,444500.0,419100.0,25300.0,2600.0
433,JOB0434,DT037,SEC004,SL003,23500.0,23400.0,100.0,100.0


In [9]:
write_table(df_final, "sc_gold", "fact_job")

Table sc_gold.fact_job written successfully.
